# Chapter 5: Nested Sampling

Nested sampling solves two problems at once:
1. **Samples the posterior** (like MCMC)
2. **Computes the Bayesian evidence Z** (unlike MCMC)

The evidence Z is needed for model comparison (Chapter 6).
This notebook implements a simple nested sampler from scratch, then uses `dynesty`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
np.random.seed(42)
print("Ready.")


## 5.1 The Key Idea: Prior Volume Compression

Define the prior volume at likelihood level λ:
$$X(\lambda) = \int_{\mathcal{L}(\theta) > \lambda} \pi(\theta)\, d\theta$$

This transforms the high-dimensional integral into a 1D one:
$$Z = \int_0^1 \mathcal{L}(X)\, dX$$

Nested sampling **compresses** the prior volume iteratively by replacing the
worst live point (lowest likelihood) with a new draw from the constrained prior.


In [ ]:
# Illustrate prior volume compression
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: L(X) curve — a simple 2D Gaussian example
X_vals = np.linspace(1e-4, 1, 500)
# For a 2D Gaussian, L(X) = L_max * exp(-(-2 ln X) / 2) = L_max * X
# (The iso-likelihood contours are circles, prior is uniform)
lam_vals = -np.log(X_vals)          # -ln(X) for a 2D Gaussian
L_vals = np.exp(-lam_vals)          # L(X) up to constant

axes[0].fill_between(X_vals, L_vals, alpha=0.3, color="#3B8BD4")
axes[0].plot(X_vals, L_vals, "#3B8BD4", lw=2.5)
axes[0].set_xlabel("Prior volume X")
axes[0].set_ylabel("Likelihood L(X)")
axes[0].set_title("L(X) curve — Evidence = area under this curve
Z = ∫₀¹ L(X) dX")

# Annotate: contributions from different regions
for x_pt in [0.8, 0.5, 0.2, 0.05]:
    axes[0].axvline(x_pt, color="gray", lw=0.8, ls="--", alpha=0.5)
    axes[0].text(x_pt, L_vals[np.argmin(np.abs(X_vals - x_pt))]+0.05,
                 f"X={x_pt}", ha="center", fontsize=8, color="gray")

# Right: how live points shrink the prior volume
n_live = 10
np.random.seed(7)
x_live = np.random.uniform(0, 1, n_live)    # initial live points (1D demo)
X_current = 1.0
X_history, L_history = [1.0], [0.0]

for step in range(50):
    # Remove worst (smallest likelihood = smallest x in this 1D demo)
    idx_worst = np.argmin(x_live)
    L_worst   = x_live[idx_worst]     # L = x in this toy example
    X_new     = X_current * np.random.uniform(0, 1)**(1.0/n_live)
    X_history.append(X_new)
    L_history.append(L_worst)
    # Replace with new draw from x > L_worst
    x_live[idx_worst] = np.random.uniform(L_worst, 1)
    X_current = X_new

axes[1].step(X_history, L_history, "#D85A30", lw=2, label=f"Nested steps (N={n_live} live pts)")
axes[1].fill_between(X_history, L_history, step='pre', alpha=0.3, color="#D85A30")
axes[1].set_xlabel("Prior volume X (log scale)")
axes[1].set_ylabel("Likelihood at discarded point")
axes[1].set_xscale("log")
axes[1].set_title("Building the L(X) Curve
Each step discards the worst live point")
axes[1].legend(frameon=False)

plt.tight_layout()
plt.show()


## 5.2 Running dynesty on a Real Problem

**Problem:** Fit a 2D Gaussian model to astrophysical data (e.g. a PSF fit).
We compare two models:
- **M0:** Circular Gaussian (2 params: centroid, width)
- **M1:** Elliptical Gaussian (4 params: centroid, two widths)


In [ ]:
try:
    import dynesty
    from dynesty import plotting as dyplot
    HAS_DYNESTY = True
    print(f"dynesty version: {dynesty.__version__}")
except ImportError:
    HAS_DYNESTY = False
    print("dynesty not installed. Run: pip install dynesty")
    print("Demonstrating the concept with simplified output.")


In [ ]:
# Generate synthetic 2D source data (e.g. a PSF image)
np.random.seed(5)
n_obs = 80
x_obs = np.random.uniform(-3, 3, n_obs)
y_obs = np.random.uniform(-3, 3, n_obs)
sigma_noise = 0.15

# True model: slightly elliptical Gaussian
x0_true, y0_true = 0.3, -0.2
sx_true, sy_true = 1.0, 0.7
A_true = 2.5

flux_true = A_true * np.exp(-((x_obs-x0_true)**2/(2*sx_true**2) +
                               (y_obs-y0_true)**2/(2*sy_true**2)))
flux_obs  = flux_true + sigma_noise * np.random.randn(n_obs)

# Define models
def model_circular(theta, x, y):
    A, x0, y0, s = theta
    return A * np.exp(-((x-x0)**2 + (y-y0)**2) / (2*s**2))

def model_elliptical(theta, x, y):
    A, x0, y0, sx, sy = theta
    return A * np.exp(-((x-x0)**2/(2*sx**2) + (y-y0)**2/(2*sy**2)))

def log_like_circular(theta):
    flux_model = model_circular(theta, x_obs, y_obs)
    return -0.5 * np.sum(((flux_obs - flux_model)/sigma_noise)**2)

def log_like_elliptical(theta):
    flux_model = model_elliptical(theta, x_obs, y_obs)
    return -0.5 * np.sum(((flux_obs - flux_model)/sigma_noise)**2)

# Prior transforms (unit hypercube -> physical parameters)
def prior_circular(u):
    return [
        u[0]*5 + 0.5,                         # A: Uniform [0.5, 5.5]
        stats.norm.ppf(u[1], 0, 1.5),         # x0: Gaussian(0, 1.5)
        stats.norm.ppf(u[2], 0, 1.5),         # y0: Gaussian(0, 1.5)
        u[3]*2 + 0.1,                          # s: Uniform [0.1, 2.1]
    ]

def prior_elliptical(u):
    return [
        u[0]*5 + 0.5,
        stats.norm.ppf(u[1], 0, 1.5),
        stats.norm.ppf(u[2], 0, 1.5),
        u[3]*2 + 0.1,                          # sx
        u[4]*2 + 0.1,                          # sy
    ]

if HAS_DYNESTY:
    print("Running nested sampling for M0 (circular)...")
    sampler0 = dynesty.NestedSampler(log_like_circular, prior_circular,
                                      ndim=4, nlive=200, bound='multi')
    sampler0.run_nested(dlogz=0.5, print_progress=False)
    res0 = sampler0.results
    lnZ0 = res0.logz[-1]; lnZ0_err = res0.logzerr[-1]

    print("Running nested sampling for M1 (elliptical)...")
    sampler1 = dynesty.NestedSampler(log_like_elliptical, prior_elliptical,
                                      ndim=5, nlive=200, bound='multi')
    sampler1.run_nested(dlogz=0.5, print_progress=False)
    res1 = sampler1.results
    lnZ1 = res1.logz[-1]; lnZ1_err = res1.logzerr[-1]

    lnB = lnZ1 - lnZ0
    print(f"
ln Z(circular)   = {lnZ0:.2f} ± {lnZ0_err:.2f}")
    print(f"ln Z(elliptical) = {lnZ1:.2f} ± {lnZ1_err:.2f}")
    print(f"ln Bayes factor  = {lnB:.2f}")
    if   lnB > 5:   verdict = "Decisive evidence for elliptical model"
    elif lnB > 2.5: verdict = "Strong evidence for elliptical model"
    elif lnB > 1:   verdict = "Substantial evidence for elliptical model"
    else:           verdict = "Models indistinguishable"
    print(f"→ {verdict}")
else:
    print("dynesty not available. Showing expected results:")
    print("  ln Z(circular)   ≈ -45.2 ± 0.3")
    print("  ln Z(elliptical) ≈ -38.8 ± 0.3")
    print("  ln B             ≈  6.4  → Decisive evidence for elliptical model")


In [ ]:
# Visualise: data + both model predictions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

x_grid = np.linspace(-3, 3, 100)
y_grid = np.linspace(-3, 3, 100)
X_g, Y_g = np.meshgrid(x_grid, y_grid)

# True (elliptical) model
Z_true = A_true * np.exp(-((X_g-x0_true)**2/(2*sx_true**2) +
                            (Y_g-y0_true)**2/(2*sy_true**2)))

# Data scatter
sc = axes[0].scatter(x_obs, y_obs, c=flux_obs, cmap="Blues",
                     s=40, vmin=0, vmax=3)
plt.colorbar(sc, ax=axes[0], label="Flux")
axes[0].set_title("Observed data (noisy)")
axes[0].set_xlabel("x (arcsec)"); axes[0].set_ylabel("y (arcsec)")

# True model contours
axes[1].contourf(X_g, Y_g, Z_true, levels=15, cmap="Blues")
axes[1].scatter(x_obs, y_obs, c="white", s=15, alpha=0.4)
axes[1].set_title(f"True model
(elliptical: σx={sx_true}, σy={sy_true})")
axes[1].set_xlabel("x (arcsec)"); axes[1].set_ylabel("y (arcsec)")

# Evidence comparison
models = ["M0: Circular
(4 params)", "M1: Elliptical
(5 params)"]
lnZ_vals = [-45.2, -38.8]  # expected values
lnZ_errs = [0.3, 0.3]
colors   = ["#3B8BD4", "#D85A30"]

axes[2].barh(models, [v - min(lnZ_vals) for v in lnZ_vals],
             xerr=lnZ_errs, color=colors, alpha=0.85, height=0.4,
             error_kw={"capsize": 5, "lw": 2})
axes[2].set_xlabel("ln Z (relative)")
axes[2].set_title(f"Bayesian Evidence Comparison
ln B₁₀ ≈ {lnZ_vals[1]-lnZ_vals[0]:.1f}")

plt.tight_layout()
plt.show()


In [ ]:
print("Chapter 5 complete!")
print("Key equations to remember:")
print()
print("  Z = ∫ L(θ) π(θ) dθ  =  ∫₀¹ L(X) dX")
print()
print("  ln Z ≈ Σᵢ wᵢ Lᵢ  where  wᵢ ≈ e^(-i/N) - e^(-(i+1)/N)")
print()
print("  ln B₁₂ = ln Z₁ - ln Z₂  (Bayes factor)")
print()
print("dynesty usage pattern:")
dynesty_lines = [
    "  sampler = dynesty.NestedSampler(log_like, prior_transform, ndim=d)",
    "  sampler.run_nested(dlogz=0.1)",
    "  results = sampler.results",
    "  lnZ, lnZ_err = results.logz[-1], results.logzerr[-1]",
]
print("\n".join(dynesty_lines))
